## Music Alignment

Music alignment links different representations of the *same* piece of music, so that a position in one
representation can be mapped to the corresponding position in another.

This notebook covers two offline alignment tasks:

- **Audio-to-audio / audio-to-score alignment** with **Sync Toolbox** — chroma features and MrMsDTW produce a
  *warping path* between two timelines https://github.com/meinardmueller/synctoolbox
- **MIDI-to-score alignment** with **parangonar** — a *note-level* alignment that matches individual score notes
  to individual performed notes https://github.com/sildater/parangonar

The two operate at different granularities. Sync Toolbox aligns *time axes*: it answers "second 12.4 of this
recording corresponds to second 9.8 of that one." Parangonar aligns *note events*: it answers "this printed note
was played as that MIDI note," and can also report notes that were skipped or added.

All examples use Chopin's Étude Op. 10 No. 3 from the `resources/` folder — two different performances of the
same excerpt, the engraved score, and MIDI transcriptions produced by the models in
`2_automatic_music_transcription.ipynb`.

## 0. Environment Setup

Run the setup cell once at the beginning. It works both in **Google Colab** and on a **local** machine:

- In Colab it clones this repository (which carries the audio, score and MIDI files used throughout) and installs
  the alignment packages.
- Locally it only installs the packages, assuming the notebook is already inside a clone of the repository. If you
  prefer a conda environment: `conda create -n ksmpc python=3.11 && conda activate ksmpc`.

If the dependencies are already installed you may skip the installation cell.

In [ ]:
# Setup: works both in Google Colab and locally.
import importlib.util
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB and not Path("ksmpc2026").exists() and not Path("../resources").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/laurenceyoon/ksmpc2026.git"],
        check=True,
    )

%pip install -q synctoolbox librosa libtsm parangonar partitura pretty_midi pandas matplotlib

# synctoolbox's feature.visualization module is not part of the PyPI 1.4.2 release, so the GitHub
# copy is installed on top of it. Both report version 1.4.2, which would make pip treat the
# requirement as already satisfied -- hence the explicit --force-reinstall.
%pip install -q --force-reinstall --no-deps "synctoolbox @ git+https://github.com/groupmm/synctoolbox.git"

print(f"Running in {'Colab' if IN_COLAB else 'a local environment'}. Installation complete.")

In [ ]:
# Imports used throughout the notebook.
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import IPython.display as ipd

from synctoolbox.dtw.mrmsdtw import sync_via_mrmsdtw
from synctoolbox.dtw.utils import (
    compute_optimal_chroma_shift,
    shift_chroma_vectors,
    make_path_strictly_monotonic,
)
from synctoolbox.feature.chroma import pitch_to_chroma, quantize_chroma, quantized_chroma_to_CENS
from synctoolbox.feature.pitch import audio_to_pitch_features
from synctoolbox.feature.utils import estimate_tuning
from synctoolbox.feature.visualization import plot_chromagram, plot_signal

%matplotlib inline

# Global parameters, shared by every alignment in this notebook.
Fs = 22050              # sampling rate
feature_rate = 50       # feature frames per second
step_weights = np.array([1.5, 1.5, 2.0])
threshold_rec = 10 ** 6
figsize = (9, 3)

# Locate the repository root: the parent directory when this notebook runs from inside the
# clone (notebooks/), or the freshly cloned ksmpc2026/ when running in Colab.
PROJECT_ROOT = next(
    (p.resolve() for p in (Path(".."), Path("ksmpc2026"), Path(".")) if (p / "resources").is_dir()),
    None,
)
assert PROJECT_ROOT is not None, "Could not find the repository's resources/ directory."
RESOURCE_DIR = PROJECT_ROOT / "resources"

# Two different performances of the same Chopin excerpt.
PERFORMANCE_AUDIOS = {
    "p09": RESOURCE_DIR / "Chopin_op10_no3_p09_short.wav",
    "p15": RESOURCE_DIR / "Chopin_op10_no3_p15_short.wav",
}
# The musical score: engraved image, MusicXML (note-level), and a deadpan MIDI rendering.
SCORE_IMAGE = RESOURCE_DIR / "Chopin_op10_no3_p15_short.png"
SCORE_MUSICXML = RESOURCE_DIR / "Chopin_op10_no3_short.musicxml"
SCORE_MIDI = RESOURCE_DIR / "Chopin_op10_no3_short_score.mid"

# Transcribed performance MIDI files (produced in 2_automatic_music_transcription.ipynb).
PERFORMANCE_MIDIS = {
    "p09 (ai-midi)": RESOURCE_DIR / "Chopin_op10_no3_p09_short_ai-midi.mid",
    "p15 (ai-midi)": RESOURCE_DIR / "Chopin_op10_no3_p15_short_ai-midi.mid",
    "p15 (basic-pitch)": RESOURCE_DIR / "Chopin_op10_no3_p15_short_basicpitch.mid",
}

OUTPUT_DIR = PROJECT_ROOT / "results" / "music_alignment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [*PERFORMANCE_AUDIOS.values(), SCORE_MUSICXML, SCORE_MIDI, *PERFORMANCE_MIDIS.values()]:
    assert path.is_file(), f"Missing resource: {path}"
print(f"Setup complete. Resources found in {RESOURCE_DIR}")

# 1. Audio-to-Audio Alignment with Sync Toolbox

Two pianists never play a piece at exactly the same speed. Alignment finds the time-warping that maps one
performance onto the other, so that corresponding musical events line up.

This section uses two performances of Chopin's Étude Op. 10 No. 3 (`p09` and `p15`) and follows the Sync Toolbox
pipeline: estimate tuning, compute chroma features, detect transposition, align with MrMsDTW, then sonify.

While listening, consider the following questions.

- Which performance is faster overall, and does that difference stay constant?
- Where do the two performances differ most in timing?

In [ ]:
display(ipd.Image(filename=str(SCORE_IMAGE), width=900))

audios = {}
for name, path in PERFORMANCE_AUDIOS.items():
    audio, _ = librosa.load(path, sr=Fs)
    audios[name] = audio
    print(f"{name}: {path.name} ({len(audio) / Fs:.1f} seconds)")
    plot_signal(audio, Fs=Fs, ylabel="Amplitude", title=f"Performance {name}", figsize=figsize)
    plt.show()
    display(ipd.Audio(audio, rate=Fs))

audio_1, audio_2 = audios["p09"], audios["p15"]

## 1.1 Estimating Tuning

Recordings are often tuned slightly away from A=440 Hz. The deviation is estimated per recording and passed to the
feature extraction. Without this correction the chroma features become smeared across neighbouring pitch classes,
which degrades the alignment.

In [ ]:
tuning_offset_1 = estimate_tuning(audio_1, Fs)
tuning_offset_2 = estimate_tuning(audio_2, Fs)
print(f"Estimated tuning deviation - p09: {tuning_offset_1} cents, p15: {tuning_offset_2} cents")

## 1.2 Computing Chroma Features

Chroma features collapse the spectrum onto the 12 pitch classes. This discards timbre and octave register while
keeping the harmonic content, which is what makes two different performances comparable.

MrMsDTW internally smooths, downsamples and normalizes these features, so the quantized chroma is passed directly.

In [ ]:
def get_chroma_from_audio(audio, tuning_offset, visualize=False):
    """Compute quantized chroma features for one audio signal."""
    f_pitch = audio_to_pitch_features(
        f_audio=audio,
        Fs=Fs,
        tuning_offset=tuning_offset,
        feature_rate=feature_rate,
        verbose=visualize,
    )
    f_chroma = pitch_to_chroma(f_pitch=f_pitch)
    return quantize_chroma(f_chroma=f_chroma)


f_chroma_quantized_1 = get_chroma_from_audio(audio_1, tuning_offset_1)
f_chroma_quantized_2 = get_chroma_from_audio(audio_2, tuning_offset_2)

plot_chromagram(f_chroma_quantized_1, Fs=feature_rate, title="Chroma - performance p09", figsize=figsize)
plt.show()
plot_chromagram(f_chroma_quantized_2, Fs=feature_rate, title="Chroma - performance p15", figsize=figsize)
plt.show()

## 1.3 Detecting Transposition

Two recordings may be played in different keys, which circularly shifts one chroma sequence relative to the other
and would break the alignment. All 12 shifts are tried on coarse CENS features and the lowest-cost one is kept.

For two performances of the same edition the shift is normally 0, but the check makes the pipeline robust to any
pair of recordings.

In [ ]:
f_cens_1hz_1 = quantized_chroma_to_CENS(f_chroma_quantized_1, 201, 50, feature_rate)[0]
f_cens_1hz_2 = quantized_chroma_to_CENS(f_chroma_quantized_2, 201, 50, feature_rate)[0]
opt_chroma_shift = compute_optimal_chroma_shift(f_cens_1hz_1, f_cens_1hz_2)
print(f"Optimal chroma shift (p15 relative to p09): {opt_chroma_shift} semitone bins")

f_chroma_quantized_2 = shift_chroma_vectors(f_chroma_quantized_2, opt_chroma_shift)

## 1.4 Aligning with MrMsDTW

**MrMsDTW** (multi-resolution multi-scale DTW) aligns the two chroma sequences on progressively finer resolutions,
each level constrained by the alignment found on the level above. This keeps the computation fast and
memory-efficient enough for complete pieces, instead of filling an N x M cost matrix.

The result is a **warping path** `wp`: index pairs `[n, m]` stating that frame *n* of p09 corresponds to frame
*m* of p15.

Setting `verbose=True` makes Sync Toolbox draw its own diagnostic plots — the cost matrices of each resolution
level with the warping path and the anchor points laid over them.

In [ ]:
wp = sync_via_mrmsdtw(
    f_chroma1=f_chroma_quantized_1,
    f_chroma2=f_chroma_quantized_2,
    input_feature_rate=feature_rate,
    step_weights=step_weights,
    threshold_rec=threshold_rec,
    verbose=True,
)
plt.show()

### Making the Warping Path Strictly Monotonic

The standard DTW step sizes allow horizontal and vertical steps, so the raw path can briefly stand still in one
recording while the other advances. For sonification and annotation transfer a **strictly monotonic** path is
cleaner, interpolating linearly through the non-monotonic segments.

In [ ]:
print(f"Warping path length (raw): {wp.shape[1]}")
wp = make_path_strictly_monotonic(wp)
print(f"Warping path length (strictly monotonic): {wp.shape[1]}")

### Plotting the Warping Path

The warping path is plotted in seconds. A straight diagonal would mean the two performances run at an identical
tempo; deviations from the diagonal are exactly where the performances disagree in timing.

The lower panel shows the accumulated time difference between the two performances along the path.

In [ ]:
wp_seconds = wp / feature_rate
t_p09, t_p15 = wp_seconds[0], wp_seconds[1]

fig, axes = plt.subplots(2, 1, figsize=(7, 8), height_ratios=[3, 1], constrained_layout=True)

axes[0].plot(t_p09, t_p15, color="crimson", linewidth=2, label="Warping path")
diagonal = [0, min(t_p09[-1], t_p15[-1])]
axes[0].plot(diagonal, diagonal, color="gray", linestyle="--", linewidth=1, label="Identical tempo")
axes[0].set_xlabel("Time in p09 (seconds)")
axes[0].set_ylabel("Time in p15 (seconds)")
axes[0].set_title("Warping path between the two performances")
axes[0].legend()
axes[0].grid(alpha=0.2)
axes[0].set_aspect("equal")

axes[1].plot(t_p09, t_p15 - t_p09, color="royalblue", linewidth=2)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_xlabel("Time in p09 (seconds)")
axes[1].set_ylabel("p15 - p09 (s)")
axes[1].set_title("Accumulated time difference")
axes[1].grid(alpha=0.2)
plt.show()

print(f"Total duration - p09: {len(audio_1) / Fs:.1f} s, p15: {len(audio_2) / Fs:.1f} s")

## 1.5 Sonifying the Alignment

The clearest test of an alignment is to listen to it. Performance p09 is time-scaled along the warping path so
that it runs synchronously with p15, then the two are placed in opposite stereo channels. If the alignment is
correct the performances stay locked together throughout.

Note that time-scale modification only alters the signal being warped — the reference keeps its original audio,
so any stretching artifacts land on p09.

In [ ]:
import libtsm

# Pitch-shift p09 to match p15's key (a no-op when the chroma shift is 0).
pitch_shift_for_audio_1 = -opt_chroma_shift % 12
if pitch_shift_for_audio_1 > 6:
    pitch_shift_for_audio_1 -= 12
audio_1_shifted = libtsm.pitch_shift(audio_1, pitch_shift_for_audio_1 * 100, order="tsm-res")

# libtsm expects the warping path in audio samples.
time_map = wp.T / feature_rate * Fs
time_map[time_map[:, 0] > len(audio_1), 0] = len(audio_1) - 1
time_map[time_map[:, 1] > len(audio_2), 1] = len(audio_2) - 1

y_hpstsm = libtsm.hps_tsm(audio_1_shifted, time_map)

# The warped signal can differ by a sample or two; trim both to the common length.
L = min(len(audio_2), y_hpstsm.shape[0])
stereo_sonification = np.column_stack((audio_2[:L], y_hpstsm[:L].reshape(-1)))

print("Original p09")
display(ipd.Audio(audio_1, rate=Fs, normalize=True))
print("Original p15")
display(ipd.Audio(audio_2, rate=Fs, normalize=True))
print("Synchronized: p15 reference (left) + p09 time-warped (right)")
display(ipd.Audio(stereo_sonification.T, rate=Fs, normalize=True))

## 1.6 Exporting the Alignment

The warping path is saved for use outside the notebook.

- **`warping_path_p09_p15.csv`** — the full path as corresponding times in the two recordings.
- **`sonic_visualiser_*.csv`** — sparse correspondence layers for
  [Sonic Visualiser](https://www.sonicvisualiser.org/). Open one recording, then
  *File > Import Annotation Layer...* and pick the matching CSV. Each point is labelled with the corresponding
  time in the other recording.

In [ ]:
pd.DataFrame({"time_p09_sec": t_p09, "time_p15_sec": t_p15}).to_csv(
    OUTPUT_DIR / "warping_path_p09_p15.csv", index=False
)
print(f"Wrote warping_path_p09_p15.csv with {wp.shape[1]} points")

# Sparse correspondence layers, one point per second.
sv_interval_sec = 1.0
for name, (own, other) in {"p09": (t_p09, t_p15), "p15": (t_p15, t_p09)}.items():
    grid = np.arange(own[0], own[-1], sv_interval_sec)
    pd.DataFrame({
        "time": grid,
        "label": np.round(np.interp(grid, own, other), 3),
    }).to_csv(OUTPUT_DIR / f"sonic_visualiser_{name}.csv", index=False, header=False)
    print(f"Wrote sonic_visualiser_{name}.csv")

# 2. Audio-to-Score Alignment

The same machinery aligns a **recording** to a **score**. The score is first rendered to audio, which reduces the
problem to the audio-to-audio case already solved above — the score rendering supplies a reference timeline in
which every note position is known exactly.

The result maps score time onto performance time, which is what drives score-following applications: page turning,
automatic accompaniment, and transferring score annotations onto a recording.

In [ ]:
import pretty_midi

# Render the score MIDI to audio. pretty_midi's built-in sine synthesis needs no SoundFont,
# and chroma features are indifferent to the timbre anyway.
score_pm = pretty_midi.PrettyMIDI(str(SCORE_MIDI))
score_audio = score_pm.synthesize(fs=Fs).astype(np.float32)
performance_audio = audios["p15"]

print(f"Score rendering: {len(score_audio) / Fs:.1f} seconds")
display(ipd.Audio(score_audio, rate=Fs, normalize=True))
print(f"Performance p15: {len(performance_audio) / Fs:.1f} seconds")
display(ipd.Audio(performance_audio, rate=Fs, normalize=True))

In [ ]:
f_chroma_score = get_chroma_from_audio(score_audio, estimate_tuning(score_audio, Fs))
f_chroma_perf = get_chroma_from_audio(performance_audio, estimate_tuning(performance_audio, Fs))

plot_chromagram(f_chroma_score, Fs=feature_rate, title="Chroma - score rendering", figsize=figsize)
plt.show()
plot_chromagram(f_chroma_perf, Fs=feature_rate, title="Chroma - performance p15", figsize=figsize)
plt.show()

The chroma of the score rendering is visibly cleaner than the performance chroma: no pedal blur, no room
reverberation, and every note at a constant velocity. Alignment tolerates this mismatch because chroma compares
*which* pitch classes sound, not how they sound.

In [ ]:
wp_score = sync_via_mrmsdtw(
    f_chroma1=f_chroma_score,
    f_chroma2=f_chroma_perf,
    input_feature_rate=feature_rate,
    step_weights=step_weights,
    threshold_rec=threshold_rec,
    verbose=False,
)
wp_score = make_path_strictly_monotonic(wp_score)
wp_score_seconds = wp_score / feature_rate
t_score, t_perf = wp_score_seconds[0], wp_score_seconds[1]
print(f"Score-to-performance warping path: {wp_score.shape[1]} points")

### Transferring Score Note Onsets onto the Recording

The warping path turns any score time into a performance time. Applying it to every note onset in the score gives
the moment each printed note is heard in the recording — score annotations transferred onto audio.

In [ ]:
score_onsets_sec = np.array([note.start for inst in score_pm.instruments for note in inst.notes])
score_pitches = np.array([note.pitch for inst in score_pm.instruments for note in inst.notes])
order = np.argsort(score_onsets_sec)
score_onsets_sec, score_pitches = score_onsets_sec[order], score_pitches[order]

# Map score onsets onto the performance timeline.
predicted_onsets_sec = np.interp(score_onsets_sec, t_score, t_perf)

print(f"Transferred {len(score_onsets_sec)} score onsets onto the performance timeline.")
for i in range(5):
    print(f"  score {score_onsets_sec[i]:6.2f}s (pitch {score_pitches[i]:3d}) "
          f"-> performance {predicted_onsets_sec[i]:6.2f}s")

The transferred onsets are drawn as click markers over the performance waveform, and sonified as clicks mixed with
the recording. If the alignment is right the clicks land on the notes.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)

axes[0].plot(t_score, t_perf, color="crimson", linewidth=2, label="Warping path")
diagonal = [0, min(t_score[-1], t_perf[-1])]
axes[0].plot(diagonal, diagonal, color="gray", linestyle="--", linewidth=1, label="Identical tempo")
axes[0].set_xlabel("Time in score rendering (seconds)")
axes[0].set_ylabel("Time in performance p15 (seconds)")
axes[0].set_title("Score-to-performance warping path")
axes[0].legend()
axes[0].grid(alpha=0.2)

times = np.arange(len(performance_audio)) / Fs
axes[1].plot(times, performance_audio, color="lightsteelblue", linewidth=0.5)
axes[1].vlines(predicted_onsets_sec, -1, 1, color="crimson", alpha=0.5, linewidth=0.8)
axes[1].set_xlim(0, times[-1])
axes[1].set_xlabel("Time (seconds)")
axes[1].set_ylabel("Amplitude")
axes[1].set_title("Performance p15 with transferred score onsets")
plt.show()

clicks = librosa.clicks(times=predicted_onsets_sec, sr=Fs, length=len(performance_audio))
print("Performance with transferred score onsets as clicks")
display(ipd.Audio(performance_audio + 0.5 * clicks, rate=Fs, normalize=True))

# 3. MIDI-to-Score Alignment with parangonar

Sync Toolbox aligns *timelines*. **parangonar** aligns *note events*: it matches each note in the score to the
note that realizes it in a performance MIDI, and reports notes present in only one of the two.

This is a strictly harder problem than time warping. A performance may roll a chord, ornament a passage, skip a
note, or add one — so the mapping is not a one-to-one correspondence in time order. Parangonar therefore labels
every note as one of:

- **match** — a score note paired with a performed note
- **deletion** — a score note that was never played
- **insertion** — a performed note absent from the score

The performance MIDI files here were transcribed from audio in `2_automatic_music_transcription.ipynb`, so
insertions and deletions include transcription errors as well as genuine performance decisions.

In [ ]:
import partitura as pt
import parangonar as pa

print(f"partitura {pt.__version__}, parangonar {pa.__version__}")

# The score is loaded from MusicXML, which carries the notated information
# (voices, measures, note ids) that the MIDI rendering does not.
score = pt.load_score(str(SCORE_MUSICXML))
score_note_array = pt.utils.music.ensure_notearray(score, include_grace_notes=True)
print(f"Score: {len(score_note_array)} notes")

performances = {}
for name, path in PERFORMANCE_MIDIS.items():
    performance = pt.load_performance_midi(str(path))
    performances[name] = performance
    print(f"{name}: {len(performance.note_array())} notes ({path.name})")

## 3.1 Running the Note Matcher

`AutomaticNoteMatcher` performs note-level alignment without requiring any manual anchor points. It first
aligns the two sequences coarsely, then matches individual notes inside each aligned region.

In [ ]:
matcher = pa.AutomaticNoteMatcher()

alignments = {}
for name, performance in performances.items():
    alignment = matcher(score_note_array, performance.note_array())
    alignments[name] = alignment

    labels = pd.Series([note["label"] for note in alignment]).value_counts()
    matched = labels.get("match", 0)
    print(f"{name}: {matched} matches, "
          f"{labels.get('deletion', 0)} deletions, {labels.get('insertion', 0)} insertions "
          f"({matched / len(score_note_array):.1%} of the score matched)")

## 3.2 Visualizing the Note Alignment

Parangonar draws the score notes and the performed notes as two piano rolls, with a line connecting every matched
pair. The slope of the connecting lines shows the local tempo relationship, and unconnected notes are the
insertions and deletions.

In [ ]:
target = "p15 (ai-midi)"
pa.plot_alignment(
    performances[target].note_array(),
    score_note_array,
    alignments[target],
    fname=f"Note alignment - score vs {target}",
)
plt.show()

## 3.3 Comparing Two Matchers

Parangonar ships several matching algorithms. `DualDTWNoteMatcher` uses a different strategy from
`AutomaticNoteMatcher`, so running both on the **same** performance shows how much the choice of algorithm
matters.

`plot_alignment_comparison` overlays the two results: notes both matchers agree on are drawn in one colour, and
the disagreements stand out.

In [ ]:
dual_matcher = pa.DualDTWNoteMatcher()
alignment_dual = dual_matcher(score_note_array, performances[target].note_array())

labels_dual = pd.Series([note["label"] for note in alignment_dual]).value_counts()
print(f"DualDTWNoteMatcher:      {labels_dual.get('match', 0)} matches, "
      f"{labels_dual.get('deletion', 0)} deletions, {labels_dual.get('insertion', 0)} insertions")

labels_auto = pd.Series([note["label"] for note in alignments[target]]).value_counts()
print(f"AutomaticNoteMatcher:    {labels_auto.get('match', 0)} matches, "
      f"{labels_auto.get('deletion', 0)} deletions, {labels_auto.get('insertion', 0)} insertions")

In [ ]:
pa.plot_alignment_comparison(
    performances[target].note_array(),
    score_note_array,
    alignments[target],
    alignment_dual,
    figsize=(30, 10),
)
plt.show()

### Quantifying the Agreement

Treating the `AutomaticNoteMatcher` result as the reference, the `DualDTWNoteMatcher` result is scored with
precision, recall and F-score over the matched pairs. Neither is ground truth, so this measures *agreement*
between the two algorithms rather than absolute correctness.

In [ ]:
precision, recall, f_score = pa.fscore_alignments(
    alignment_dual,
    alignments[target],
    types=["match"],
)
print("DualDTWNoteMatcher vs AutomaticNoteMatcher (matched notes)")
print(f"  Precision: {precision:.3f}")
print(f"  Recall:    {recall:.3f}")
print(f"  F-score:   {f_score:.3f}")

### How Transcription Quality Affects Alignment

The three performance MIDI files were produced by transcription models of differing accuracy. The proportion of
score notes that each one lets the matcher recover is a practical measure of how transcription errors propagate
into alignment.

In [ ]:
summary = []
for name, alignment in alignments.items():
    labels = pd.Series([note["label"] for note in alignment]).value_counts()
    summary.append({
        "performance": name,
        "performed notes": len(performances[name].note_array()),
        "match": labels.get("match", 0),
        "deletion": labels.get("deletion", 0),
        "insertion": labels.get("insertion", 0),
        "score coverage": f"{labels.get('match', 0) / len(score_note_array):.1%}",
    })
display(pd.DataFrame(summary))

## 3.4 Extracting Performance Timing

Because the alignment pairs individual notes, the notated onset of every matched note can be read next to the
second at which it was actually played. This is the raw material for performance analysis — the mapping from
notated beat to performed time.

In [ ]:
def alignment_to_dataframe(alignment, score_note_array, performance_note_array):
    """Collect matched note pairs into a DataFrame of notated vs. performed timing."""
    score_by_id = {str(note["id"]): note for note in score_note_array}
    performance_by_id = {str(note["id"]): note for note in performance_note_array}

    rows = []
    for note in alignment:
        if note["label"] != "match":
            continue
        score_note = score_by_id.get(str(note["score_id"]))
        performance_note = performance_by_id.get(str(note["performance_id"]))
        if score_note is None or performance_note is None:
            continue
        rows.append({
            "score_id": str(note["score_id"]),
            "onset_beat": score_note["onset_beat"],
            "pitch": performance_note["pitch"],
            "onset_sec": performance_note["onset_sec"],
            "duration_sec": performance_note["duration_sec"],
            "velocity": performance_note["velocity"],
        })
    return pd.DataFrame(rows).sort_values("onset_beat").reset_index(drop=True)


timing = alignment_to_dataframe(
    alignments[target], score_note_array, performances[target].note_array()
)
display(timing.head(10))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
scatter = ax.scatter(
    timing["onset_beat"], timing["onset_sec"],
    c=timing["velocity"], cmap="magma", s=25,
)
ax.set_xlabel("Notated onset (beats)")
ax.set_ylabel("Performed onset (seconds)")
ax.set_title(f"Notated versus performed onsets - {target}")
ax.grid(alpha=0.2)
fig.colorbar(scatter, ax=ax, label="MIDI velocity")
plt.show()

The slope of this curve is the local tempo: steeper segments are slower playing, flatter segments faster. The
departures from a straight line are the performer's rubato.

## 3.5 Exporting the Alignment

Parangonar writes the alignment in the formats used by the wider ecosystem. The `match` file is the standard
note-alignment format, and the Parangonada CSV bundle can be inspected interactively at
[https://sildater.github.io/parangonada/](https://sildater.github.io/parangonada/).

The `resources/alignment/` folder of this repository holds exports produced this way.

In [ ]:
for name, alignment in alignments.items():
    export_dir = OUTPUT_DIR / name.replace(" ", "_").replace("(", "").replace(")", "")
    export_dir.mkdir(parents=True, exist_ok=True)

    pt.io.exportparangonada.save_parangonada_csv(
        alignment,
        performances[name],
        score,
        outdir=str(export_dir),
    )
    pt.save_match(
        alignment=alignment,
        performance_data=performances[name],
        score_data=score,
        out=str(export_dir / "alignment.match"),
    )
    print(f"{name} -> {export_dir}")

## 3.6 Collecting the Results

Everything this notebook wrote is listed below. In Colab the runtime's filesystem is discarded when the session
ends, so uncomment the download lines to keep the exports.

In [ ]:
print(f"Generated files in {OUTPUT_DIR}:")
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT_DIR)} ({path.stat().st_size:,} bytes)")

# In Colab, uncomment to download the alignment exports to your computer.
# from google.colab import files
# for path in sorted(OUTPUT_DIR.glob("*.csv")):
#     files.download(str(path))

# 4. Try It Yourself

- **Swap the reference.** In section 1, align p15 onto p09 instead. The warping path mirrors about the diagonal,
  and the time-stretch artifacts move to the other recording.
- **Change the step weights.** `step_weights = np.array([1.0, 1.0, 1.0])` removes the preference for diagonal
  steps, making the path more willing to stretch time. Compare the warping paths.
- **Align across transcriptions.** Section 3 aligns each transcription to the score; try aligning the two
  transcriptions of p15 to each other and see which notes only one model found.
- **Use your own recording.** Any pair of performances of the same piece works in section 1, and any performance
  MIDI of this Chopin excerpt works in section 3.

# References

[1] Meinard Müller, Yigitcan Özer, Michael Krause, Thomas Prätzlich, and Jonathan Driedger: Sync Toolbox: A Python
Package for Efficient, Robust, and Accurate Music Synchronization, JOSS, 2021.

[2] Thomas Prätzlich, Jonathan Driedger, and Meinard Müller: Memory-Restricted Multiscale Dynamic Time Warping,
ICASSP, 2016.

[3] Meinard Müller, Henning Mattes, and Frank Kurth: An Efficient Multiscale Approach to Audio Synchronization,
ISMIR, 2006.

[4] Silvan David Peter, Carlos Cancino-Chacón, Francesco Foscarin, Andrew McLeod, Florian Henkel, Emmanouil
Karystinaios, and Gerhard Widmer: Automatic Note-Level Score-to-Performance Alignments in the ASAP Dataset,
TISMIR, 2023.

[5] Carlos Cancino-Chacón, Silvan David Peter, Emmanouil Karystinaios, Francesco Foscarin, Maarten Grachten, and
Gerhard Widmer: Partitura: A Python Package for Symbolic Music Processing, MEC, 2022.